# Answer Generation (v2 — 1 LLM Call Total)
**What changed from v1:**
- ❌ Old: 4 LLM calls per query (traverse × 2, re-rank × 1, answer × 1)
- ✅ New: 1 LLM call per query (answer only)
- ✅ `retrieve()` imported from `retreivalPDF.ipynb` logic — uses 0 LLM calls
- ✅ `rank_pages()` removed entirely

Pipeline:
```
query
  -> retrieve()             (0 LLM calls — keyword similarity routing)
  -> generate_answer()      (1 LLM call — answer from page content only)
```

**Run order:** `pageMetadata.ipynb` → `treeBuilder.ipynb` → `answerGeneration.ipynb`

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


## 1. Imports & Connection

In [2]:
import json
import re
import time

from src.config.db import get_connection
from src.config.llm import llm

conn = get_connection()


## 2. Set Document

In [3]:
document_id = "DOC000001"


## 3. Retrieval Pipeline (from retreivalPDF v2 — 0 LLM calls)

In [4]:
class TreeNode:
    def __init__(self, data: dict, parent=None):
        self.data     = data
        self.parent   = parent
        self.children = []

    @property
    def level(self):    return self.data.get("level", 0)
    @property
    def title(self):    return self.data.get("title", "")
    @property
    def summary(self):  return self.data.get("summary", "")
    @property
    def keywords(self): return self.data.get("keywords", [])
    @property
    def path(self):     return self.data.get("path", "")
    @property
    def type(self):     return self.data.get("type", "root")

    def __repr__(self):
        return f"TreeNode(path={self.path!r}, title={self.title!r})"


def build_tree_nodes(tree_dict: dict, parent=None) -> "TreeNode":
    node = TreeNode(tree_dict, parent)
    for child_dict in tree_dict.get("children", []):
        node.children.append(build_tree_nodes(child_dict, parent=node))
    return node


def load_tree_from_db(conn, document_id: str) -> "TreeNode":
    with conn.cursor() as cur:
        cur.execute(
            'SELECT "treeJson" FROM "Tree" WHERE "documentId" = %s',
            (document_id,),
        )
        row = cur.fetchone()
    if row is None:
        raise ValueError(f"No tree found for documentId={document_id!r}")
    tree_dict = row[0]
    if isinstance(tree_dict, str):
        tree_dict = json.loads(tree_dict)
    return build_tree_nodes(tree_dict)


def keyword_overlap(query: str, node: "TreeNode") -> float:
    query_words = set(re.sub(r"[^a-z0-9 ]", " ", query.lower()).split())
    node_text   = " ".join([node.summary, " ".join(node.keywords), node.title]).lower()
    node_words  = set(re.sub(r"[^a-z0-9 ]", " ", node_text).split())
    overlap     = query_words & node_words
    return len(overlap) / (len(query_words) + 1e-9)


def traverse_tree(query: str, root: "TreeNode") -> dict:
    best_chapter = max(root.children,    key=lambda ch:  keyword_overlap(query, ch))
    best_section = max(best_chapter.children, key=lambda sec: keyword_overlap(query, sec))
    return {"path": [root, best_chapter, best_section], "leaf": best_section}


def get_candidate_pages(conn, leaf: "TreeNode") -> list:
    page_ids = leaf.data.get("pageIds", [])
    if not page_ids:
        return []
    placeholders = ",".join(["%s"] * len(page_ids))
    with conn.cursor() as cur:
        cur.execute(
            f'SELECT "pageNumber", metadata FROM "Page" WHERE id IN ({placeholders}) ORDER BY "pageNumber"',
            page_ids,
        )
        rows = cur.fetchall()
    candidates = []
    for page_number, metadata in rows:
        meta = metadata or {}
        candidates.append({
            "pageNumber": page_number,
            "title":      meta.get("title", ""),
            "summary":    meta.get("summary", ""),
            "keywords":   meta.get("keywords", []),
        })
    return candidates


def fetch_pages_content(conn, document_id: str, page_numbers: list) -> dict:
    if not page_numbers:
        return {}
    placeholders = ",".join(["%s"] * len(page_numbers))
    with conn.cursor() as cur:
        cur.execute(
            f'SELECT "pageNumber", content FROM "Page" WHERE "documentId" = %s AND "pageNumber" IN ({placeholders})',
            (document_id, *page_numbers),
        )
        rows = cur.fetchall()
    content_by_page = {pn: content for pn, content in rows}
    return {p: content_by_page[p] for p in page_numbers if p in content_by_page}


def retrieve(query: str, document_id: str, conn, top_k: int = 5) -> dict:
    """Zero-LLM retrieval — keyword similarity routing only."""
    tree_root     = load_tree_from_db(conn, document_id)
    traversal     = traverse_tree(query, tree_root)
    leaf          = traversal["leaf"]
    candidates    = get_candidate_pages(conn, leaf)
    page_numbers  = [c["pageNumber"] for c in candidates]
    top_pages     = page_numbers[:top_k]
    pages_content = fetch_pages_content(conn, document_id, top_pages)
    return {
        "tree_path":     [{"type": n.type, "title": n.title, "path": n.path} for n in traversal["path"]],
        "leaf":          leaf.data,
        "candidates":    candidates,
        "ranked_pages":  page_numbers,
        "top_pages":     top_pages,
        "pages_content": pages_content,
    }


## 4. Answer Generation Prompt (unchanged)
The only LLM call in the entire query pipeline.

In [5]:
ANSWER_PROMPT = """You are answering a question using ONLY the context pages below.
This is a Vectorless RAG system — there is no other source of truth.

Rules:
1. Answer using only the provided context. Do not use outside knowledge.
2. If the answer is not contained in the context, say so explicitly —
   do not guess or make anything up.
3. When you use information from a page, cite it inline as (Page N).
4. Be concise and directly answer the question first, then add supporting detail.

Question:
{query}

Context:
{context}

Answer:
"""


## 5. Generate Answer Function (unchanged)

In [6]:
def build_context(pages_content: dict) -> str:
    blocks = []
    for page_number, text in pages_content.items():
        blocks.append(f"========== Page {page_number} ==========\n{text}")
    return "\n\n".join(blocks)


def generate_answer(query: str, pages_content: dict, llm) -> str:
    """Single LLM call — the only one in the entire query pipeline."""
    if not pages_content:
        return "I couldn't find any relevant pages in the document to answer this question."
    context = build_context(pages_content)
    prompt  = ANSWER_PROMPT.format(query=query, context=context)
    return llm.invoke(prompt).content.strip()


## 6. Full Answer Pipeline
`retrieve()` = 0 LLM calls, `generate_answer()` = 1 LLM call. Total = **1**.

In [7]:
def answer_query(query: str, document_id: str, conn, llm, top_k: int = 5) -> dict:
    """
    Full pipeline: retrieve (0 LLM calls) → generate answer (1 LLM call).

    Returns:
      {
        "query":           str,
        "answer":          str,
        "sources":         [page_number, ...],
        "tree_path":       [...],
        "latency_seconds": float,
        "llm_calls":       int,   # always 1
      }
    """
    start = time.time()

    retrieval = retrieve(query, document_id, conn, top_k=top_k)
    answer    = generate_answer(query, retrieval["pages_content"], llm)

    elapsed = time.time() - start

    return {
        "query":           query,
        "answer":          answer,
        "sources":         retrieval["top_pages"],
        "tree_path":       retrieval["tree_path"],
        "latency_seconds": round(elapsed, 2),
        "llm_calls":       1,
    }


## 7. Demo

In [8]:
query = "What Summarize Monitoring Financial Vulnerabilities?"

result = answer_query(query, document_id, conn, llm, top_k=5)

print("Query:", result["query"])
print()
print("Answer:")
print(result["answer"])
print()
print("Sources (pages):", result["sources"])
print("Latency:        ", result["latency_seconds"], "s")
print("LLM calls:      ", result["llm_calls"], "(down from 4)")


Query: What Summarize Monitoring Financial Vulnerabilities?

Answer:
The Federal Reserve monitors four key vulnerabilities to financial stability: leverage in the financial sector, funding risk, borrowing by businesses and households, and asset valuations (Page 22). 

These vulnerabilities are monitored to better understand the complex linkages between financial institutions, households, and businesses, and to promote financial stability by informing broader policy discussions and stimulating additional research (Page 22). The Federal Reserve assesses these vulnerabilities quarterly to identify potential risks to financial stability and to mitigate the consequences of financial instability (Page 21).

Sources (pages): [21, 22]
Latency:         0.8 s
LLM calls:       1 (down from 4)


## 8. Interactive Query (optional, manual testing)

In [9]:
question = input("Enter your question: ")

result = answer_query(question, document_id, conn, llm, top_k=5)

print("\nAnswer:")
print(result["answer"])
print("\nSources (pages):", result["sources"])
print("LLM calls this query:", result["llm_calls"])



Answer:
Monitoring Financial Vulnerabilities refers to the Federal Reserve's efforts to identify and assess potential risks to financial stability (Page 21). This involves periodically assessing four key vulnerabilities: leverage in the financial sector, funding risk, borrowing by businesses and households, and asset valuations (Page 22). The goal of monitoring these vulnerabilities is to promote financial stability by informing broader policy discussions and stimulating additional research, ultimately helping to maintain a resilient financial system (Page 21).

Sources (pages): [21, 22]
LLM calls this query: 1
